# 05 - Agrupamento com groupby

## Objetivo
Agrupar dados por uma ou mais chaves e aplicar agregacoes.

## Conceitos

### O padrao split-apply-combine
O `groupby` segue o padrao:
1. Split: divide os dados em grupos por chave.
2. Apply: aplica uma funcao a cada grupo.
3. Combine: junta os resultados.

### Agregacoes comuns
- `sum`, `mean`, `median`, `min`, `max`, `count`, `std`, `var`.
- `agg` permite varias funcoes ao mesmo tempo.
- `agg` com dicionario permite funcao diferente por coluna.

### Agrupamento multiplo
`df.groupby(["col1", "col2"])` cria grupos por combinacao de chaves.

### Transform e filter
- `transform`: retorna resultado do mesmo tamanho do original.
- `filter`: remove grupos inteiros com base em uma condicao.
- `apply`: aplica funcao arbitraria a cada grupo.

### Tabela dinamica
`pd.pivot_table` combina groupby com reshape, similar a planilhas.

### Reset do indice
Apos agrupar, o resultado tem indice multi-nivel.
Use `reset_index()` para voltar a um DataFrame plano.

## DataFrame de exemplo
Criamos um DataFrame com departamento, cidade, funcionario, salario e idade
para praticar os agrupamentos a seguir.

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "departamento": ["TI", "RH", "TI", "Vendas", "RH", "TI", "Vendas"],
    "cidade": ["SP", "RJ", "SP", "MG", "SP", "RJ", "SP"],
    "funcionario": ["Ana", "Bruno", "Carla", "Diego", "Elisa",
                    "Fabio", "Gabi"],
    "salario": [5000, 3800, 6200, 4500, 4100, 5500, 4700],
    "idade": [28, 35, 31, 29, 42, 26, 33],
})
print("DataFrame:\n", df)

## Agrupamento simples
`groupby("coluna")` divide o DataFrame em grupos. Ao selecionar uma coluna
numerica e aplicar uma agregacao (ex.: `mean`), obtemos uma Series
indexada pelo valor da chave de agrupamento.

In [ ]:
# Agrupamento simples
por_depto = df.groupby("departamento")
print("\nMedia por departamento:\n",
      por_depto["salario"].mean())

## Multiplas agregacoes
- `agg(["mean", "min", "max", "count"])` aplica varias funcoes de uma vez.
- Com dicionario, define funcao diferente por coluna.

In [ ]:
# Multiplas agregacoes
print("\nAgregacoes por departamento:\n",
      por_depto["salario"].agg(["mean", "min", "max", "count"]))

# agg com dicionario
print("\nagg por coluna:\n",
      por_depto.agg({"salario": ["mean", "max"],
                     "idade": "mean"}))

## Agrupamento por multiplas chaves
`groupby(["col1", "col2"])` cria grupos para cada combinacao de chaves.
O resultado tem indice **MultiIndex**.

In [ ]:
# Agrupamento por multiplas chaves
multi = df.groupby(["departamento", "cidade"])["salario"].mean()
print("\nAgrupamento multiplo:\n", multi)

## Reset do indice
Apos agrupar por multiplas chaves, o indice tem varios niveis.
`reset_index()` volta ao formato tabular plano, com uma coluna para
cada chave.

In [ ]:
# Reset do indice
multi_reset = multi.reset_index()
print("\nResetado:\n", multi_reset)

# Contagem de registros por grupo
print("\nContagem:\n", por_depto.size())

## transform
`transform` aplica uma funcao a cada grupo, mas retorna um resultado
do **mesmo tamanho** do DataFrame original. Util para criar colunas
com estatisticas do grupo em cada linha.

In [ ]:
# transform: media do grupo em cada linha
df["media_depto"] = df.groupby("departamento")["salario"].transform("mean")
print("\nCom media do departamento:\n", df)

# Diferenca em relacao a media do grupo
df["diff_media"] = df["salario"] - df["media_depto"]
print("\nDiferenca em relacao a media:\n",
      df[["funcionario", "departamento", "salario", "diff_media"]])

## filter
`filter(cond)` mantem apenas os **grupos inteiros** que satisfazem
a condicao. Grupos descartados somem por completo.

In [ ]:
# filter: manter apenas grupos com mais de 2 membros
grupos_grandes = df.groupby("departamento").filter(lambda g: len(g) > 2)
print("\nGrupos com mais de 2 membros:\n", grupos_grandes)

## apply
`apply(funcao)` permite aplicar uma funcao **arbitraria** a cada grupo,
retornando um resultado consolidado. No exemplo, calculamos a amplitude
salarial (max - min) por departamento.

In [ ]:
# apply: funcao customizada
def amplitude(g):
    return g["salario"].max() - g["salario"].min()

print("\nAmplitude salarial por departamento:\n",
      por_depto.apply(amplitude, include_groups=False))

## pivot_table e crosstab
- `pivot_table`: agrega valores em uma tabela cruzada (linhas x colunas),
  similar a planilhas. Combina `groupby` com reshape.
- `crosstab`: calcula a **frequencia** de combinacoes entre duas colunas
  categoricas.

In [ ]:
# pivot_table
pivot = pd.pivot_table(df, values="salario",
                       index="departamento", columns="cidade",
                       aggfunc="mean")
print("\npivot_table:\n", pivot)

# crosstab: frequencia
print("\ncrosstab:\n",
      pd.crosstab(df["departamento"], df["cidade"]))